# Template
Copy this notebook to start a new lesson. Edit only the config cell to scale up on the 5090.

In [1]:
# --- Config: bump these on the 5090 (e.g. BS=128, IMG_SIZE=448) ---
BS = 32          # fits 8 GB on the 3070 Ti
IMG_SIZE = 224
FP16 = True      # use learn.to_fp16() when True

In [ ]:
import os
from pathlib import Path

# Project root = parent of nbs/. Keep fastai downloads inside the git-ignored data/ folder.
PROJECT = Path.cwd().resolve().parent if Path.cwd().name == 'nbs' else Path.cwd().resolve()
DATA = Path(os.environ.get('LOCAL_AI_DATA', PROJECT / 'data'))
MODELS = Path(os.environ.get('LOCAL_AI_MODELS', PROJECT / 'models'))
os.environ.setdefault('FASTAI_HOME', str(DATA / 'fastai'))  # must be set before importing fastai

from fastai.vision.all import *
import torch
print(torch.cuda.get_device_name(0), '|', 'BS', BS, '| IMG_SIZE', IMG_SIZE, '| FP16', FP16)

In [ ]:
from ddgs import DDGS #DuckDuckGo has changed the api so we need to update 
from fastcore.all import *

def search_images(keywords, max_images=200): return L(DDGS().images(keywords, max_results=max_images)).itemgot('image')
import time, json

In [ ]:
#NB: `search_images` depends on duckduckgo.com, which doesn't always return correct responses.
#    If you get a JSON error, just try running it again (it may take a couple of tries).
urls = search_images('bird photos', max_images=1)
urls[0]

In [ ]:
from fastdownload import download_url
dest = 'bird.jpg'
download_url(urls[0], dest, show_progress=False)

from fastai.vision.all import *
im = Image.open(dest)
im.to_thumb(256,256)

In [ ]:
download_url(search_images('forest photos', max_images=1)[0], 'forest.jpg', show_progress=False)
Image.open('forest.jpg').to_thumb(256,256)

In [ ]:
searches = 'forest','bird'
path = Path('bird_or_not')

for o in searches:
    dest = (path/o)
    dest.mkdir(exist_ok=True, parents=True)
    download_images(dest, urls=search_images(f'{o} photo'))
    time.sleep(5)
    resize_images(path/o, max_size=400, dest=path/o)

In [ ]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink)
len(failed)

In [3]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock), 
    get_items=get_image_files, 
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=32)

dls.show_batch(max_n=6)


NameError: name 'DataBlock' is not defined

In [12]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/mario/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:03<00:00, 13.0MB/s]


epoch,train_loss,valid_loss,error_rate,time
0,1.239776,1.477058,0.357143,00:03


epoch,train_loss,valid_loss,error_rate,time
0,0.796013,1.206201,0.285714,00:01
1,0.712814,0.967704,0.357143,00:01
2,0.518804,0.705992,0.285714,00:01


In [13]:
is_bird,_,probs = learn.predict(PILImage.create('bird.jpg'))
print(f"This is a: {is_bird}.")
print(f"Probability it's a bird: {probs[0]:.4f}")

This is a: bird.
Probability it's a bird: 1.0000
